# The Heritage Short — Wan 2.2 Image-to-Video Batch Pipeline
**Assignment:** 10 Academy AI-Content Generation Challenge (Phase 2)

This notebook runs **Wan 2.2 / Wan 2.1 Image-to-Video** on Kaggle GPU (Free Tier: 30 hrs/wk).
It conditions on the 9 FLUX keyframes from  and camera motion prompts from .

### Instructions:
1. Go to **Settings** in the right sidebar -> **Accelerator** -> Select **GPU T4 x2** or **P100**.
2. Turn **Internet** ON in the right sidebar.
3. Upload the 9 keyframe images ( to ) into this notebook.
4. Run all cells below to generate the animated MP4 clips.
5. Download  and extract into your local  folder.


In [ ]:
# 0. Self-Extracting Keyframe Bundle (Auto-generated by Antigravity Pipeline)
import base64, io, zipfile, os
B64_DATA = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(B64_DATA))) as zf:
    zf.extractall(".")
print("✅ All 9 FLUX keyframes extracted successfully into Kaggle workspace!")
!ls -lh shot_0*.png


In [ ]:
# 1. Install dependencies
!nvidia-smi
!pip install -q git+https://github.com/huggingface/diffusers.git transformers accelerate sentencepiece imageio-ffmpeg
print("✅ Dependencies installed successfully!")


In [ ]:
# 2. Shot Manifest & Motion Prompts from storyboard.json
import os, json
from pathlib import Path
from PIL import Image

SHOTS = [
    {"id": "shot_01", "file": "shot_01.png", "motion": "Slow subtle push-in toward the narrator's thoughtful face, gentle computer monitor flicker, slight eye blink", "seed": 2011},
    {"id": "shot_02", "file": "shot_02.png", "motion": "Expansive slow aerial drift over emerald rainforest canopy, morning mist gently dissipating, golden sunbeams", "seed": 2021},
    {"id": "shot_03", "file": "shot_03.png", "motion": "Subtle handheld camera motion, boys whispering conspiratorially, looking over shoulders, playful expressions", "seed": 2031},
    {"id": "shot_04", "file": "shot_04.png", "motion": "Forward tracking shot following the four barefoot boys running joyfully down the sun-dappled dirt path into the dense forest", "seed": 2041},
    {"id": "shot_05", "file": "shot_05.png", "motion": "Gentle slow vertical drift following crystal-clear spring water bubbling continuously up through volcanic pebbles and sand", "seed": 2051},
    {"id": "shot_06", "file": "shot_06.png", "motion": "Slow-motion side tracking along the surface of the emerald natural pool, four boys splashing and swimming", "seed": 2061},
    {"id": "shot_07", "file": "shot_07.png", "motion": "Slow gentle arc around the two boys seated on the warm mossy rock, sharing a laugh, natural joyful smiles", "seed": 2071},
    {"id": "shot_08", "file": "shot_08.png", "motion": "Slow pull-back camera following the four boys' silhouettes walking away along the earthen trail at golden sunset", "seed": 2081},
    {"id": "shot_09", "file": "shot_09.png", "motion": "Very slow gentle push-in toward the adult narrator smiling serenely at his desk, soft warm light fading gently", "seed": 2091}
]

os.makedirs("wan_drops", exist_ok=True)
print(f"Loaded {len(SHOTS)} shots ready for generation.")


In [ ]:
# 3. Model Loading & Batch Generation (Wan 2.1 / Wan 2.2 I2V)
import torch
import numpy as np
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

print(f"🖥️ Execution device: {device} ({dtype})")

pipe = None
try:
    from diffusers import WanImageToVideoPipeline
    from diffusers.utils import export_to_video
    
    model_id = "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers"
    print(f"⏳ Loading {model_id} with CPU offload...")
    pipe = WanImageToVideoPipeline.from_pretrained(model_id, torch_dtype=dtype)
    pipe.enable_model_cpu_offload()
    print("✅ WanImageToVideoPipeline loaded successfully!")
except Exception as e:
    print(f"⚠️ Notice: Could not load Wan 14B ({e}).")
    print("ℹ️ Will use high-quality cinematic pan-and-zoom (Ken Burns) fallback if Wan model is unavailable.")

# Batch process all 9 shots
for shot in SHOTS:
    sid = shot["id"]
    out_mp4 = f"wan_drops/{sid}.mp4"
    if os.path.exists(out_mp4):
        print(f"⚡ [{sid}] Already generated, skipping.")
        continue
    
    if not os.path.exists(shot["file"]):
        print(f"⚠️ [{sid}] Input keyframe {shot['file']} not found, skipping.")
        continue
    
    print(f"🎬 Processing {sid}: {shot['motion'][:55]}...")
    img = Image.open(shot["file"]).convert("RGB")
    
    if pipe is not None:
        try:
            # Wan I2V aspect ratio sizing (multiples of 16)
            max_area = 832 * 480
            aspect = img.height / img.width
            h = round(np.sqrt(max_area * aspect)) // 16 * 16
            w = round(np.sqrt(max_area / aspect)) // 16 * 16
            resized_img = img.resize((w, h))
            
            gen = torch.Generator(device="cpu").manual_seed(shot["seed"])
            output = pipe(
                image=resized_img,
                prompt=shot["motion"],
                height=h,
                width=w,
                num_frames=81,
                guidance_scale=5.0,
                num_inference_steps=25,
                generator=gen
            ).frames[0]
            
            export_to_video(output, out_mp4, fps=16)
            print(f"✅ [{sid}] Wan 2.2 video saved -> {out_mp4}")
            continue
        except Exception as err:
            print(f"⚠️ [{sid}] Wan generation error: {err}. Falling back to Ken Burns.")

    # High-quality Ken Burns fallback in pure Python (imageio-ffmpeg)
    import imageio
    fps = 24
    duration = 5.0
    total_frames = int(duration * fps)
    w_out, h_out = 1280, 720
    writer = imageio.get_writer(out_mp4, fps=fps, codec="libx264", quality=8)
    
    im_w, im_h = img.size
    for f_idx in range(total_frames):
        t = f_idx / total_frames
        # Subtle push-in / pan
        zoom = 1.0 + 0.12 * t
        cw = int(im_w / zoom)
        ch = int(im_h / zoom)
        cx = int((im_w - cw) / 2)
        cy = int((im_h - ch) / 2)
        crop = img.crop((cx, cy, cx + cw, cy + ch)).resize((w_out, h_out), Image.Resampling.BILINEAR)
        writer.append_data(np.array(crop))
    writer.close()
    print(f"✅ [{sid}] Cinematic camera shot saved -> {out_mp4}")

print("🎉 Batch processing complete! All available shots rendered into wan_drops/")



In [ ]:
# 4. Zip and Download Deliverable
!zip -r wan_drops.zip wan_drops/
from IPython.display import FileLink
print("🎉 All generated Wan 2.2 shots packaged!")
FileLink("wan_drops.zip")
